# 05 COF 性质预测：一个完整 baseline

使用仓库内置教学数据 `data/cof_demo.csv` 演示 preprocessing、categorical encoding、grouped split 和 Random Forest。`CO2_uptake_demo` 是人工构造标签，不能用于科研结论。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
url='https://raw.githubusercontent.com/Wanteen/COF-ML-Tutorial/main/data/cof_demo.csv'
df=pd.read_csv(url)
df.head()

In [ ]:
target='CO2_uptake_demo'
features=['family','functional_group','pore_A','void_fraction','density','N_fraction','O_fraction']
X=df[features]; y=df[target]
cat=['family','functional_group']; num=[c for c in features if c not in cat]
pre=ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
model=Pipeline([('pre',pre),('rf',RandomForestRegressor(n_estimators=500,random_state=42,n_jobs=-1))])

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42)
model.fit(X_train,y_train); pred=model.predict(X_test)
print('Random split')
print('MAE',mean_absolute_error(y_test,pred),'RMSE',mean_squared_error(y_test,pred)**0.5,'R2',r2_score(y_test,pred))

In [ ]:
splitter=GroupShuffleSplit(n_splits=1,test_size=0.25,random_state=42)
train_idx,test_idx=next(splitter.split(X,y,groups=df['family']))
X_train2,X_test2=X.iloc[train_idx],X.iloc[test_idx]; y_train2,y_test2=y.iloc[train_idx],y.iloc[test_idx]
model.fit(X_train2,y_train2); pred2=model.predict(X_test2)
print('Train families:',sorted(df.iloc[train_idx]['family'].unique()))
print('Test families:',sorted(df.iloc[test_idx]['family'].unique()))
print('MAE',mean_absolute_error(y_test2,pred2),'RMSE',mean_squared_error(y_test2,pred2)**0.5,'R2',r2_score(y_test2,pred2))

## 关键讨论
如果 random split 很好，而 family-aware split 明显变差，说明模型擅长在见过的化学空间附近插值，并不代表能外推到新的 COF family。

## 必做题
1. 删除 functional group，比较性能。
2. 删除 pore features，比较性能。
3. 尝试 GradientBoosting / XGBoost。
4. 解释 random split 与 family-aware split 的差异。
5. 设计真正可用于 COF screening 的评价方案。